# Live ranking validation
Runs 10 queries against real agreed input tables and returns a compact validation report.
No input tables are changed. Missing inputs fail explicitly.


In [ ]:
import json
import math
from pathlib import Path
from time import perf_counter
from ranking import load_from_spark

started = perf_counter()
ranker = load_from_spark(spark)
queries = json.loads(Path("examples/04_ranking_queries.json").read_text())
summary = []
for query in queries:
    start = perf_counter()
    result = ranker.recommend(query["service"], query["latitude"], query["longitude"], k=5)
    assert result["status"] != "unknown_service", query
    assert result["eligible_candidate_count"] > 0, result["excluded_rows"]
    nearby = result["recommendations"]
    assert all(c["distance_km"] <= result["policy"]["nearby_km"] for c in nearby)
    assert len({c["unique_id"] for c in nearby}) == len(nearby)
    assert [c["score"] for c in nearby] == sorted([c["score"] for c in nearby], reverse=True)
    nearest = result["nearest_strong_option"]
    cards = nearby + ([nearest] if nearest else [])
    for candidate in cards:
        assert ranker.facilities[candidate["unique_id"]].get("flag_row_corrupted") is not True
        p = candidate["score_breakdown"]
        assert math.isclose(candidate["score"], p["evidence_score"] * p["location_weight"] * p["distance_factor"], rel_tol=1e-12)
        assert candidate["quotes"]
        assert not any("unavailable for text verification" in warning for warning in candidate["data_warnings"])
    summary.append({
        "location": query["location"], "service": result["service"], "status": result["status"],
        "eligible_candidates": result["eligible_candidate_count"], "nearby_count": len(nearby),
        "nearby_weak_count": len(result["nearby_weak_options"]),
        "top_id": nearby[0]["unique_id"] if nearby else None,
        "top_name": nearby[0]["name"] if nearby else None,
        "top_score": nearby[0]["score"] if nearby else None,
        "nearest_strong_id": nearest["unique_id"] if nearest else None,
        "nearest_strong_km": round(nearest["distance_km"], 2) if nearest else None,
        "excluded_rows": result["excluded_rows"], "seconds": round(perf_counter() - start, 3),
    })
report = {
    "status": "passed", "facility_rows": len(ranker.facilities),
    "evidence_rows": sum(len(rows) for rows in ranker.evidence.values()),
    "services": sorted(ranker.evidence), "queries_passed": len(summary),
    "total_seconds": round(perf_counter() - started, 3), "queries": summary,
}
print(json.dumps(report, indent=2, allow_nan=False))
dbutils.notebook.exit(json.dumps(report, allow_nan=False))
